# Redes Neuronales V — Reconocimiento facial con Redes Neuronales Siamesas y Triplet Loss

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jerson09/redes-neuronales-v/blob/main/siamese_face_recognition.ipynb)

**Framework:** TensorFlow 2 + Keras 3 · **Entorno:** Google Colab (GPU T4)

Este notebook implementa, de extremo a extremo, un sistema de **verificación e identificación facial** basado en una
**red neuronal siamesa** entrenada con **Triplet Loss**:

1. Marco conceptual (redes siamesas, embeddings, Triplet Loss, One/Few/Zero-shot).
2. Carga del dataset propio (imágenes con consentimiento) y control de calidad.
3. Preprocesamiento facial: corrección EXIF, detección con MTCNN, alineación por ojos, recorte, normalización de iluminación (CLAHE) y redimensionado.
4. Datos públicos de entrenamiento (LFW) y generación de tripletas (ancla, positivo, negativo).
5. Arquitectura: red de embeddings (MobileNetV2 + proyección 128-D normalizada L2) compartida por tres ramas.
6. Entrenamiento con Triplet Loss.
7. Evaluación sobre el dataset propio: verificación (ROC/AUC), One-shot, Few-shot, identidades no vistas (zero-shot) y rechazo de desconocidos.
8. Función de comparación de dos rostros y exportación de resultados.

> **Antes de ejecutar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

## 1. Marco conceptual

### 1.1 Redes Neuronales Siamesas
Una red siamesa está formada por **dos o más ramas idénticas que comparten exactamente los mismos pesos**. Cada rama
transforma una imagen en un vector de características (*embedding*) $f(x) \in \mathbb{R}^{d}$. En lugar de clasificar
la imagen en una de *N* clases fijas, la red aprende una **función de similitud**: dos fotos de la misma persona deben
producir embeddings cercanos y fotos de personas distintas, embeddings lejanos. Por eso:

* No es necesario reentrenar al añadir una persona nueva: basta con calcular su embedding (registro o *enrollment*).
* Funciona con muy pocos ejemplos por persona (uno o algunos).

En esta práctica usamos la variante de **tres ramas** (ancla, positivo, negativo), propia de Triplet Loss.

### 1.2 Embeddings faciales y comparación
El embedding se normaliza a norma L2 = 1 (vive en una hiperesfera), y la comparación se hace con la distancia
euclidiana al cuadrado $d(a,b)=\lVert f(a)-f(b)\rVert_2^2 \in [0,4]$ (equivalente a $2-2\cos\theta$). La decisión de
**verificación** es: *misma persona* si $d < \tau$, donde el umbral $\tau$ se calibra en un conjunto de validación.

### 1.3 Triplet Loss
Para cada tripleta (ancla $a$, positivo $p$ de la misma identidad, negativo $n$ de otra identidad):

$$\mathcal{L}(a,p,n)=\max\big(\lVert f(a)-f(p)\rVert_2^2-\lVert f(a)-f(n)\rVert_2^2+\alpha,\;0\big)$$

La pérdida obliga a que el negativo esté al menos un **margen** $\alpha$ más lejos del ancla que el positivo.
Las tripletas que ya cumplen la condición aportan pérdida 0 (tripletas "fáciles"); el aprendizaje proviene de las
tripletas difíciles y semi-difíciles (FaceNet, Schroff et al., 2015).

### 1.4 One-shot, Few-shot y Zero-shot Learning
| Paradigma | Ejemplos de la clase nueva disponibles | Cómo se aplica aquí |
|---|---|---|
| **One-shot** | 1 imagen por persona | Se registra 1 foto por persona; las demás se identifican por vecino más cercano. |
| **Few-shot** | *k* imágenes (2–5) | El prototipo de la persona es la media normalizada de *k* embeddings. |
| **Zero-shot** | 0 imágenes de entrenamiento de esas clases | El modelo se entrena **solo con LFW**: ninguna persona del dataset propio fue vista durante el entrenamiento; aun así el espacio de embeddings generaliza a identidades nuevas. En sentido estricto, el zero-shot clásico requiere información auxiliar (atributos o texto, p. ej. CLIP) para reconocer una clase sin ningún ejemplo; en verificación facial se usa para referirse a identidades no vistas en entrenamiento. |

### 1.5 Papel del preprocesamiento
La red debe aprender **identidad**, no fondo, pose, iluminación ni escala. Detectar, alinear y recortar el rostro de
forma consistente elimina variabilidad irrelevante; una mala detección o un recorte inconsistente degrada los
embeddings más que cualquier cambio de arquitectura.

## 2. Configuración del entorno

In [ ]:
# MTCNN: detector facial multitarea (caja + 5 puntos faciales)
!pip install -q mtcnn

In [ ]:
import os, glob, random, zipfile, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image, ImageOps

import tensorflow as tf
import keras
from keras import layers, ops

from sklearn.datasets import fetch_lfw_people
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA

SEED = 42
keras.utils.set_random_seed(SEED)   # fija semillas de Python, NumPy y TensorFlow
random.seed(SEED); np.random.seed(SEED)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---------------- Hiperparámetros y rutas ----------------
IMG_SIZE   = 160          # tamaño de entrada de la red (MobileNetV2 admite 96/128/160/192/224)
EMB_DIM    = 128          # dimensión del embedding facial
MARGIN     = 0.4          # margen alfa del Triplet Loss (distancias en [0, 4])
BATCH_SIZE = 32
EPOCHS     = 15
LR         = 1e-4
TRAIN_TRIPLETS_PER_EPOCH = 6000
VAL_TRIPLETS             = 1200

FACE_MARGIN = 0.25        # margen alrededor de la caja detectada
MIN_CONF    = 0.90        # confianza mínima de MTCNN
USE_CLAHE   = True        # ecualización adaptativa de histograma (iluminación)
MAX_SIDE    = 1024        # se reduce la foto antes de detectar (las fotos de celular son muy grandes)

BASE_DIR  = "/content/dataset"
RAW_DIR   = f"{BASE_DIR}/raw"         # raw/<persona>/<foto>.jpg
PROC_DIR  = f"{BASE_DIR}/processed"   # rostros alineados 160x160
OUT_DIR   = "/content/outputs"        # figuras, modelo y métricas para el informe
os.makedirs(OUT_DIR, exist_ok=True)

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def savefig(name):
    """Guarda la figura actual en OUT_DIR (evidencia para el informe)."""
    plt.savefig(f"{OUT_DIR}/{name}.png", dpi=130, bbox_inches="tight")

## 3. Dataset propio (imágenes con consentimiento)

**Estructura esperada** de `dataset.zip` (una carpeta por persona, nombre sin espacios):
```
dataset.zip
└── raw/
    ├── persona_01/   foto1.jpg, foto2.jpg, ...
    ├── persona_02/
    └── persona_03/
```
**Recomendaciones de captura:** mínimo **3 personas** y **8–15 fotos por persona**, con variaciones de expresión,
iluminación, ángulo (±30°), con/sin gafas y en días distintos. Un solo rostro principal por foto.

> ⚠️ Solo se deben usar fotografías propias o de personas que firmaron el consentimiento para uso académico
> (ver `docs/consentimiento_plantilla.md` en el repositorio). Las fotos **no** se suben al repositorio público.

In [ ]:
DATA_SOURCE = "upload"    # "upload": subir dataset.zip desde el equipo | "drive": leer desde Google Drive
DRIVE_ZIP   = "/content/drive/MyDrive/redes_neuronales_v/dataset.zip"

def has_images(folder):
    return any(f.lower().endswith(IMG_EXT) for f in os.listdir(folder))

def find_raw_root(base):
    """Busca la carpeta cuyas subcarpetas (personas) contienen imágenes, por si el zip tiene otra estructura."""
    for root, dirs, _ in os.walk(base):
        if "processed" in root:
            continue
        people = [d for d in dirs if not d.startswith(".") and has_images(os.path.join(root, d))]
        if len(people) >= 2:
            return root
    raise FileNotFoundError("No se encontró una carpeta con subcarpetas de imágenes por persona.")

os.makedirs(BASE_DIR, exist_ok=True)
if not (os.path.isdir(RAW_DIR) and os.listdir(RAW_DIR)):
    if DATA_SOURCE == "upload":
        from google.colab import files
        uploaded = files.upload()                       # seleccionar dataset.zip
        zip_path = next(iter(uploaded))
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        zip_path = DRIVE_ZIP
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(BASE_DIR)
    RAW_DIR = find_raw_root(BASE_DIR)

print("Carpeta de imágenes originales:", RAW_DIR)

In [ ]:
# Inventario del dataset
people = sorted(d for d in os.listdir(RAW_DIR)
                if os.path.isdir(os.path.join(RAW_DIR, d)) and not d.startswith(".")   # ignora .ipynb_checkpoints
                and has_images(os.path.join(RAW_DIR, d)))
raw_files = {p: sorted(f for f in glob.glob(os.path.join(RAW_DIR, p, "*")) if f.lower().endswith(IMG_EXT))
             for p in people}
inventory = pd.DataFrame({"persona": people, "imagenes": [len(raw_files[p]) for p in people]})
display(inventory)
print(f"Total: {inventory.imagenes.sum()} imágenes de {len(people)} personas")
assert len(people) >= 2, "Se necesitan al menos 2 personas"

# Muestra de imágenes originales (hasta 5 por persona)
n_show = 5
fig, axes = plt.subplots(len(people), n_show, figsize=(2.2 * n_show, 2.4 * len(people)), squeeze=False)
for i, p in enumerate(people):
    for j in range(n_show):
        ax = axes[i, j]; ax.axis("off")
        if j < len(raw_files[p]):
            ax.imshow(ImageOps.exif_transpose(Image.open(raw_files[p][j])).convert("RGB"))
            if j == 0: ax.set_title(p, fontsize=9, loc="left")
plt.suptitle("Dataset propio — imágenes originales", y=1.0); plt.tight_layout(); savefig("01_dataset_original"); plt.show()

## 4. Preprocesamiento facial

Pipeline aplicado a **cada** imagen (el mismo se usa en entrenamiento, evaluación e inferencia):

| Paso | Técnica | Motivo |
|---|---|---|
| 1 | Corrección de orientación EXIF y conversión a RGB | Las fotos de celular guardan la rotación en metadatos. |
| 2 | Redimensionado a lado máximo 1024 px | Acelera la detección sin perder el rostro. |
| 3 | Detección con **MTCNN** (confianza ≥ 0.90), se toma el rostro más grande | Aísla el rostro del fondo. |
| 4 | **Alineación**: rotación para que la línea de los ojos quede horizontal | Reduce la variación por inclinación de la cabeza. |
| 5 | Recorte cuadrado con margen del 25 % | Incluye contorno del rostro de forma consistente. |
| 6 | **CLAHE** sobre el canal L (espacio LAB) | Normaliza iluminación y contraste locales. |
| 7 | Redimensionado a 160×160 (interpolación por área) | Tamaño fijo de entrada de la red. |
| 8 | Escalado a [-1, 1] (capa `Rescaling` dentro del modelo) | Rango esperado por MobileNetV2. |
| 9 | Control de calidad: nitidez (varianza del Laplaciano) | Detecta fotos borrosas que contaminan el dataset. |

Durante el entrenamiento se añade **aumento de datos** (espejo horizontal, brillo y contraste aleatorios).

In [ ]:
from mtcnn import MTCNN
detector = MTCNN()

def load_rgb(path, max_side=MAX_SIDE):
    """Carga una imagen como RGB uint8 aplicando la orientación EXIF y limitando su tamaño."""
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    scale = max_side / max(img.size)
    if scale < 1:
        img = img.resize((int(img.width * scale), int(img.height * scale)), Image.LANCZOS)
    return np.array(img)

def apply_clahe(face_rgb):
    """CLAHE sobre la luminancia para normalizar iluminación sin alterar el color."""
    lab = cv2.cvtColor(face_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

def align_and_crop(img, det, size=IMG_SIZE, margin=FACE_MARGIN):
    """Rota la imagen para nivelar los ojos y recorta un cuadrado centrado en el rostro."""
    le = np.array(det["keypoints"]["left_eye"], dtype=np.float32)
    re = np.array(det["keypoints"]["right_eye"], dtype=np.float32)
    angle = np.degrees(np.arctan2(re[1] - le[1], re[0] - le[0]))
    eyes_center = tuple(map(float, (le + re) / 2))
    M = cv2.getRotationMatrix2D(eyes_center, angle, 1.0)
    rotated = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]),
                             flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    x, y, w, h = det["box"]
    cx, cy = M @ np.array([x + w / 2, y + h / 2, 1.0])       # centro de la caja tras la rotación
    half = max(w, h) * (1 + margin) / 2
    x1, y1 = int(max(cx - half, 0)), int(max(cy - half, 0))
    x2, y2 = int(min(cx + half, img.shape[1])), int(min(cy + half, img.shape[0]))
    face = rotated[y1:y2, x1:x2]
    return cv2.resize(face, (size, size), interpolation=cv2.INTER_AREA)

def preprocess_face(path):
    """Pipeline completo. Devuelve (rostro 160x160 uint8, info) o (None, motivo)."""
    img = load_rgb(path)
    dets = [d for d in detector.detect_faces(img) if d["confidence"] >= MIN_CONF]
    if not dets:
        return None, "sin rostro detectado"
    det = max(dets, key=lambda d: d["box"][2] * d["box"][3])   # rostro más grande
    face = align_and_crop(img, det)
    if USE_CLAHE:
        face = apply_clahe(face)
    sharpness = cv2.Laplacian(cv2.cvtColor(face, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()
    return face, {"confianza": det["confidence"], "n_rostros": len(dets), "nitidez": sharpness}

In [ ]:
# Procesa todo el dataset propio y guarda los rostros alineados
records, t0 = [], time.time()
for p in people:
    os.makedirs(os.path.join(PROC_DIR, p), exist_ok=True)
    for f in raw_files[p]:
        face, info = preprocess_face(f)
        rec = {"persona": p, "archivo": os.path.basename(f)}
        if face is None:
            rec.update({"ok": False, "motivo": info})
        else:
            out = os.path.join(PROC_DIR, p, os.path.splitext(os.path.basename(f))[0] + ".jpg")
            Image.fromarray(face).save(out, quality=95)
            rec.update({"ok": True, "salida": out, **info})
        records.append(rec)

prep = pd.DataFrame(records)
print(f"Procesadas {len(prep)} imágenes en {time.time() - t0:.1f} s")
print(f"Rostros válidos: {prep.ok.sum()} | descartadas: {(~prep.ok).sum()}")
if (~prep.ok).any():
    display(prep.loc[~prep.ok, ["persona", "archivo", "motivo"]])
ok = prep[prep.ok]
print("\nResumen por persona:")
display(ok.groupby("persona").agg(imagenes=("archivo", "count"), confianza_media=("confianza", "mean"),
                                  nitidez_media=("nitidez", "mean")).round(3))
blurry = ok[ok.nitidez < 50]
if len(blurry):
    print("⚠️ Posibles imágenes borrosas (varianza del Laplaciano < 50):")
    display(blurry[["persona", "archivo", "nitidez"]])
prep.to_csv(f"{OUT_DIR}/preprocesamiento.csv", index=False)

In [ ]:
# Antes / después del preprocesamiento (una muestra por persona)
fig, axes = plt.subplots(2, len(people), figsize=(2.6 * len(people), 5.4), squeeze=False)
for j, p in enumerate(people):
    row = ok[ok.persona == p].iloc[0]
    axes[0, j].imshow(load_rgb(os.path.join(RAW_DIR, p, row.archivo))); axes[0, j].set_title(p, fontsize=9)
    axes[1, j].imshow(Image.open(row.salida)); axes[1, j].set_title(f"conf={row.confianza:.2f}", fontsize=8)
    axes[0, j].axis("off"); axes[1, j].axis("off")
axes[0, 0].text(-0.15, 0.5, "Original", transform=axes[0, 0].transAxes, rotation=90, va="center")
axes[1, 0].text(-0.15, 0.5, "Alineado + CLAHE\n160×160", transform=axes[1, 0].transAxes, rotation=90, va="center")
plt.tight_layout(); savefig("02_preprocesamiento_antes_despues"); plt.show()

In [ ]:
# Carga del dataset propio preprocesado en memoria
own_names = sorted(ok.persona.unique())
name_to_id = {n: i for i, n in enumerate(own_names)}
X_own = np.stack([np.array(Image.open(f)) for f in ok.salida]).astype("float32")   # [0,255]
y_own = ok.persona.map(name_to_id).to_numpy()
print("Dataset propio:", X_own.shape, "| personas:", own_names)

## 5. Datos de entrenamiento: LFW (Labeled Faces in the Wild)

Con unas decenas de fotos propias no es posible entrenar un espacio de embeddings robusto sin sobreajustar. Por eso
la red se **entrena con LFW** (dataset académico público) y el dataset propio se reserva **íntegramente para
pruebas**: ninguna persona del dataset propio participa en el entrenamiento (escenario de identidades no vistas).

* Se usan las personas de LFW con ≥ 10 fotos (~158 identidades, ~4 300 imágenes).
* Las imágenes LFW ya están centradas y alineadas (*funneled*); se toma un recorte central de 140×140 que
  reproduce aproximadamente el encuadre de nuestro recorte MTCNN con margen, se aplica CLAHE y se lleva a 160×160.
* La partición entrenamiento/validación se hace **por identidad** (las personas de validación no se ven al entrenar).

In [ ]:
lfw = fetch_lfw_people(min_faces_per_person=10, color=True, resize=1.0,
                       slice_=(slice(55, 195), slice(55, 195)))   # recorte central 140x140 de 250x250
X_lfw = lfw.images
if X_lfw.max() <= 1.0:          # según la versión de scikit-learn viene en [0,1] o [0,255]
    X_lfw = X_lfw * 255.0
X_lfw = np.stack([apply_clahe(cv2.resize(im.astype("uint8"), (IMG_SIZE, IMG_SIZE),
                                         interpolation=cv2.INTER_AREA)) for im in X_lfw]).astype("float32")
y_lfw = lfw.target
print("LFW:", X_lfw.shape, "| identidades:", len(lfw.target_names))

# Partición por identidad: 85 % entrenamiento, 15 % validación
ids = np.random.default_rng(SEED).permutation(np.unique(y_lfw))
val_ids = ids[: int(0.15 * len(ids))]
is_val = np.isin(y_lfw, val_ids)
X_tr, y_tr = X_lfw[~is_val], y_lfw[~is_val]
X_va, y_va = X_lfw[is_val], y_lfw[is_val]
print(f"Entrenamiento: {len(X_tr)} imgs / {len(np.unique(y_tr))} ids | "
      f"Validación: {len(X_va)} imgs / {len(np.unique(y_va))} ids")

### 5.1 Generación de tripletas
Cada tripleta se construye eligiendo una identidad al azar, dos fotos distintas de ella (ancla y positivo) y una foto de
otra identidad (negativo). Las tripletas se generan **en línea** (nuevas en cada época) con `tf.data`, y el aumento de
datos se aplica de forma independiente a cada imagen.

In [ ]:
def triplet_generator(X, y, n_triplets, seed=None):
    """Genera n_triplets tripletas aleatorias (ancla, positivo, negativo)."""
    rng = np.random.default_rng(seed)
    by_id = {c: np.where(y == c)[0] for c in np.unique(y)}
    anchors_ids = [c for c, idx in by_id.items() if len(idx) >= 2]
    all_ids = np.array(list(by_id))
    def gen():
        for _ in range(n_triplets):
            c = rng.choice(anchors_ids)
            a, p = rng.choice(by_id[c], 2, replace=False)
            neg_c = rng.choice(all_ids[all_ids != c])
            n = rng.choice(by_id[neg_c])
            yield X[a], X[p], X[n]
    return gen

def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 20.0)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    return tf.clip_by_value(img, 0.0, 255.0)

def make_triplet_ds(X, y, n_triplets, training, seed=None):
    spec = tf.TensorSpec((IMG_SIZE, IMG_SIZE, 3), tf.float32)
    ds = tf.data.Dataset.from_generator(triplet_generator(X, y, n_triplets, seed),
                                        output_signature=(spec, spec, spec))
    if training:
        ds = ds.map(lambda a, p, n: (augment(a), augment(p), augment(n)), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.cache()          # validación: siempre las mismas tripletas
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_triplet_ds(X_tr, y_tr, TRAIN_TRIPLETS_PER_EPOCH, training=True)
val_ds   = make_triplet_ds(X_va, y_va, VAL_TRIPLETS, training=False, seed=SEED)

# Visualización de algunas tripletas
a, p, n = next(iter(train_ds))
fig, axes = plt.subplots(3, 6, figsize=(12, 6.2))
for j in range(6):
    for i, (img, t) in enumerate(zip((a, p, n), ("Ancla", "Positivo", "Negativo"))):
        axes[i, j].imshow(img[j].numpy().astype("uint8")); axes[i, j].axis("off")
        if j == 0: axes[i, j].set_title(t, loc="left", fontsize=10)
plt.suptitle("Tripletas de entrenamiento (con aumento de datos)"); plt.tight_layout(); savefig("03_tripletas"); plt.show()

## 6. Arquitectura de la red siamesa

**Red de embeddings** $f(\cdot)$ (compartida por las tres ramas):

```
Entrada 160×160×3 (RGB, [0,255])
 → Rescaling a [-1, 1]
 → MobileNetV2 preentrenada en ImageNet (sin capa de clasificación, Global Average Pooling → 1280)
     · capas iniciales congeladas (bordes, texturas genéricas)
     · últimos bloques ajustables (fine-tuning hacia rasgos faciales)
     · BatchNorm en modo inferencia para estabilidad con lotes pequeños
 → Dropout(0.2)
 → Dense(128)                      # proyección al espacio de embeddings
 → UnitNormalization (L2)          # embedding en la hiperesfera unitaria
```

**Red siamesa de tripletas:** tres entradas (ancla, positivo, negativo) → la **misma instancia** de $f$ →
`DistanceLayer` calcula $d(a,p)$ y $d(a,n)$ → Triplet Loss. Al compartir la instancia, los pesos son idénticos
y los gradientes de las tres ramas se acumulan sobre un único conjunto de parámetros.

*Justificación:* MobileNetV2 (convoluciones separables en profundidad + bloques residuales invertidos) ofrece buen
equilibrio entre precisión y costo (~2.3 M parámetros), lo que permite entrenar en una GPU T4 en pocos minutos.
El *transfer learning* desde ImageNet compensa el tamaño limitado de los datos.

In [ ]:
FINE_TUNE_FROM = 100   # MobileNetV2 tiene ~154 capas; se ajustan las posteriores a esta

def build_embedding_net(img_size=IMG_SIZE, emb_dim=EMB_DIM, fine_tune_from=FINE_TUNE_FROM):
    base = keras.applications.MobileNetV2(input_shape=(img_size, img_size, 3), include_top=False,
                                          weights="imagenet", pooling="avg")
    base.trainable = True
    for layer in base.layers[:fine_tune_from]:
        layer.trainable = False
    inp = keras.Input((img_size, img_size, 3), name="rostro")
    x = layers.Rescaling(1 / 127.5, offset=-1, name="escala_-1_1")(inp)
    x = base(x, training=False)                     # BatchNorm en modo inferencia
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(emb_dim, name="proyeccion")(x)
    out = layers.UnitNormalization(axis=-1, name="normalizacion_l2")(x)
    return keras.Model(inp, out, name="red_embeddings")


class DistanceLayer(layers.Layer):
    """Distancias euclidianas al cuadrado ancla-positivo y ancla-negativo."""
    def call(self, anchor, positive, negative):
        ap = ops.sum(ops.square(anchor - positive), axis=-1)
        an = ops.sum(ops.square(anchor - negative), axis=-1)
        return ap, an


embedding_net = build_embedding_net()

anchor_in   = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="ancla")
positive_in = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="positivo")
negative_in = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="negativo")
distances = DistanceLayer(name="distancias")(embedding_net(anchor_in),      # misma instancia →
                                             embedding_net(positive_in),    # pesos compartidos
                                             embedding_net(negative_in))
siamese_network = keras.Model([anchor_in, positive_in, negative_in], distances, name="red_siamesa")

embedding_net.summary()
siamese_network.summary()
try:
    keras.utils.plot_model(siamese_network, to_file=f"{OUT_DIR}/04_arquitectura_siamesa.png",
                           show_shapes=True, expand_nested=False, dpi=90)
    display(Image.open(f"{OUT_DIR}/04_arquitectura_siamesa.png"))
except Exception as e:
    print("plot_model no disponible:", e)

## 7. Implementación del Triplet Loss y entrenamiento

Se define un modelo Keras con `train_step` / `test_step` propios: calcula las distancias con la red siamesa,
aplica $\max(d(a,p)-d(a,n)+\alpha, 0)$ promediado en el lote y actualiza los pesos compartidos.
Además se registra la **exactitud de tripletas**: proporción de tripletas con $d(a,p) < d(a,n)$.

In [ ]:
def triplet_loss(ap_dist, an_dist, margin=MARGIN):
    """Triplet Loss: L = mean( max(d(a,p) - d(a,n) + margen, 0) )."""
    return ops.mean(ops.maximum(ap_dist - an_dist + margin, 0.0))


class SiameseModel(keras.Model):
    """Envuelve la red siamesa y entrena con Triplet Loss."""
    def __init__(self, siamese_network, margin=MARGIN, **kwargs):
        super().__init__(**kwargs)
        self.siamese_network = siamese_network
        self.margin = margin
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.acc_tracker = keras.metrics.Mean(name="triplet_acc")

    def call(self, inputs, training=False):
        return self.siamese_network(inputs, training=training)

    def _loss_and_acc(self, data, training):
        ap, an = self.siamese_network(data, training=training)
        loss = triplet_loss(ap, an, self.margin)
        acc = ops.mean(ops.cast(ap < an, "float32"))
        return loss, acc

    def train_step(self, data):
        with tf.GradientTape() as tape:
            loss, acc = self._loss_and_acc(data, training=True)
        weights = self.siamese_network.trainable_weights
        grads = tape.gradient(loss, weights)
        self.optimizer.apply_gradients(zip(grads, weights))
        self.loss_tracker.update_state(loss); self.acc_tracker.update_state(acc)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        loss, acc = self._loss_and_acc(data, training=False)
        self.loss_tracker.update_state(loss); self.acc_tracker.update_state(acc)
        return {m.name: m.result() for m in self.metrics}

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc_tracker]

In [ ]:
# Línea base ANTES del entrenamiento métrico: características ImageNet normalizadas (sin Triplet Loss)
def build_baseline_net():
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                                          weights="imagenet", pooling="avg")
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1 / 127.5, offset=-1)(inp)
    out = layers.UnitNormalization()(base(x, training=False))
    return keras.Model(inp, out, name="linea_base_imagenet")

baseline_net = build_baseline_net()

def embed(model, X, batch_size=64):
    return model.predict(X, batch_size=batch_size, verbose=0)

In [ ]:
siamese_model = SiameseModel(siamese_network)
siamese_model.compile(optimizer=keras.optimizers.Adam(LR))

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
]
t0 = time.time()
history = siamese_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
print(f"Tiempo de entrenamiento: {(time.time() - t0) / 60:.1f} min")

In [ ]:
h = pd.DataFrame(history.history)
h.index = h.index + 1
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
h[["loss", "val_loss"]].plot(ax=axes[0], marker="o", title="Triplet Loss")
h[["triplet_acc", "val_triplet_acc"]].plot(ax=axes[1], marker="o", title="Exactitud de tripletas  d(a,p) < d(a,n)")
for ax in axes: ax.set_xlabel("Época"); ax.grid(alpha=.3)
plt.tight_layout(); savefig("05_curvas_entrenamiento"); plt.show()
display(h.round(4))
embedding_net.save(f"{OUT_DIR}/red_embeddings.keras")

## 8. Evaluación

### 8.1 Calibración del umbral de verificación (LFW validación)
El umbral $\tau$ se elige en las **identidades de validación de LFW** maximizando el índice de Youden
(TPR − FPR) y luego se aplica **sin modificar** al dataset propio, para no sobreestimar los resultados.

In [ ]:
def pair_distances(E, y):
    """Distancias de todos los pares (i<j) y etiqueta 1 si son la misma persona."""
    D = ((E[:, None, :] - E[None, :, :]) ** 2).sum(-1)
    iu = np.triu_indices(len(y), k=1)
    return D[iu], (y[:, None] == y[None, :])[iu].astype(int)

def best_threshold(d, same):
    fpr, tpr, thr = roc_curve(same, -d)          # menor distancia = más probable misma persona
    i = np.argmax(tpr - fpr)
    return -thr[i]

def verification_metrics(d, same, tau):
    pred = (d < tau).astype(int)
    tp = ((pred == 1) & (same == 1)).sum(); tn = ((pred == 0) & (same == 0)).sum()
    fp = ((pred == 1) & (same == 0)).sum(); fn = ((pred == 0) & (same == 1)).sum()
    return {"AUC": roc_auc_score(same, -d), "exactitud": (tp + tn) / len(d),
            "TAR (sensibilidad)": tp / max(tp + fn, 1), "FAR (falsa aceptación)": fp / max(fp + tn, 1),
            "pares_misma": int(same.sum()), "pares_distinta": int((1 - same).sum())}

E_va = embed(embedding_net, X_va)
d_va, s_va = pair_distances(E_va, y_va)
TAU = best_threshold(d_va, s_va)
print(f"Umbral calibrado τ = {TAU:.3f}")
print("LFW validación (identidades no vistas):", {k: round(v, 4) for k, v in verification_metrics(d_va, s_va, TAU).items()})

### 8.2 Verificación sobre el dataset propio (identidades no vistas — *zero-shot*)
Se comparan **todos los pares** de rostros del dataset propio: línea base ImageNet vs. red siamesa entrenada con Triplet Loss.

In [ ]:
E_own  = embed(embedding_net, X_own)
E_base = embed(baseline_net, X_own)

d_own, s_own   = pair_distances(E_own, y_own)
d_base, s_base = pair_distances(E_base, y_own)
tau_base = best_threshold(*pair_distances(embed(baseline_net, X_va), y_va))

results_ver = pd.DataFrame({
    "Línea base ImageNet": verification_metrics(d_base, s_base, tau_base),
    "Siamesa + Triplet Loss": verification_metrics(d_own, s_own, TAU),
}).T
display(results_ver.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for d, s, lbl in ((d_base, s_base, "Línea base ImageNet"), (d_own, s_own, "Siamesa + Triplet Loss")):
    fpr, tpr, _ = roc_curve(s, -d)
    axes[0].plot(fpr, tpr, label=f"{lbl} (AUC={roc_auc_score(s, -d):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=.8); axes[0].set_xlabel("FAR (tasa de falsos positivos)")
axes[0].set_ylabel("TAR (tasa de verdaderos positivos)"); axes[0].set_title("Curva ROC — dataset propio"); axes[0].legend()
axes[1].hist(d_own[s_own == 1], bins=30, alpha=.65, label="Misma persona", density=True)
axes[1].hist(d_own[s_own == 0], bins=30, alpha=.65, label="Personas distintas", density=True)
axes[1].axvline(TAU, color="k", ls="--", label=f"Umbral τ={TAU:.2f}")
axes[1].set_xlabel("Distancia euclidiana² entre embeddings"); axes[1].set_title("Distribución de distancias (siamesa)")
axes[1].legend()
plt.tight_layout(); savefig("06_roc_y_distancias"); plt.show()

### 8.3 Identificación One-shot y Few-shot
Protocolo por episodios: en cada episodio se toman al azar *k* fotos por persona como **galería** (soporte) y el resto
como **consultas**. El prototipo de cada persona es la media normalizada de sus *k* embeddings y cada consulta se
asigna al prototipo más cercano. Se promedian 200 episodios.

In [ ]:
def l2n(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

def few_shot_eval(E, y, k, episodes=200, seed=SEED):
    rng = np.random.default_rng(seed)
    classes = [c for c in np.unique(y) if (y == c).sum() > k]
    accs, y_true, y_pred = [], [], []
    for _ in range(episodes):
        protos, q_idx, q_lbl = [], [], []
        for c in classes:
            idx = rng.permutation(np.where(y == c)[0])
            protos.append(l2n(E[idx[:k]].mean(0)))
            q_idx += list(idx[k:]); q_lbl += [c] * (len(idx) - k)
        P = np.stack(protos)
        dist = ((E[q_idx][:, None, :] - P[None]) ** 2).sum(-1)
        pred = np.array(classes)[dist.argmin(1)]
        accs.append((pred == np.array(q_lbl)).mean())
        y_true += q_lbl; y_pred += list(pred)
    return np.mean(accs), np.std(accs), np.array(y_true), np.array(y_pred)

min_imgs = np.bincount(y_own).min()
K_VALUES = [k for k in (1, 2, 3, 5) if k < min_imgs]
rows = []
for k in K_VALUES:
    for name, E in (("Línea base ImageNet", E_base), ("Siamesa + Triplet Loss", E_own)):
        m, s, _, _ = few_shot_eval(E, y_own, k)
        rows.append({"k (fotos de registro)": k, "modelo": name, "exactitud": m, "desv": s})
fs = pd.DataFrame(rows)
display(fs.pivot(index="k (fotos de registro)", columns="modelo", values="exactitud").round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, g in fs.groupby("modelo"):
    axes[0].errorbar(g["k (fotos de registro)"], g.exactitud, yerr=g.desv, marker="o", capsize=4, label=name)
axes[0].set_xlabel("k = fotos de registro por persona (1 = one-shot)"); axes[0].set_ylabel("Exactitud de identificación")
axes[0].set_title("One-shot / Few-shot"); axes[0].set_xticks(K_VALUES); axes[0].legend(); axes[0].grid(alpha=.3)

_, _, yt, yp = few_shot_eval(E_own, y_own, k=1)
ConfusionMatrixDisplay(confusion_matrix(yt, yp, labels=list(range(len(own_names))), normalize="true"),
                       display_labels=own_names).plot(
    ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False)
axes[1].set_title("Matriz de confusión One-shot (siamesa)"); plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right")
plt.tight_layout(); savefig("07_one_few_shot"); plt.show()

### 8.4 Reconocimiento en conjunto abierto: rechazo de personas no registradas
Un sistema real debe responder **"desconocido"** cuando la persona no está en la galería. Para cada persona se
simula que **no** está registrada (se elimina de la galería) y se mide cuántas de sus fotos son correctamente
rechazadas (distancia al prototipo más cercano ≥ τ). Para las personas registradas se mide la identificación correcta.

In [ ]:
def open_set_eval(E, y, tau, k=1, seed=SEED):
    rng = np.random.default_rng(seed)
    rej, acc = [], []
    classes = np.unique(y)
    for unknown in classes:
        gallery = {}
        for c in classes:
            if c == unknown: continue
            idx = rng.permutation(np.where(y == c)[0])
            gallery[c] = (l2n(E[idx[:k]].mean(0)), idx[k:])
        ids = list(gallery); P = np.stack([gallery[c][0] for c in ids])
        # personas desconocidas → deben ser rechazadas
        d_unk = ((E[y == unknown][:, None] - P[None]) ** 2).sum(-1).min(1)
        rej.append((d_unk >= tau).mean())
        # personas registradas → deben aceptarse con la identidad correcta
        for c in ids:
            q = gallery[c][1]
            if len(q) == 0: continue
            dq = ((E[q][:, None] - P[None]) ** 2).sum(-1)
            acc.append(((dq.min(1) < tau) & (np.array(ids)[dq.argmin(1)] == c)).mean())
    return np.mean(rej), np.mean(acc)

open_rows = []
for k in K_VALUES:
    r, a = open_set_eval(E_own, y_own, TAU, k=k)
    open_rows.append({"k": k, "rechazo correcto de desconocidos": r, "aceptación+identificación correcta": a})
open_df = pd.DataFrame(open_rows)
display(open_df.round(4))

### 8.5 Visualización del espacio de embeddings
Proyección PCA 2D de los embeddings del dataset propio: con la red siamesa cada persona debería formar un grupo
compacto y separado de los demás.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (E, title) in zip(axes, ((E_base, "Línea base ImageNet"), (E_own, "Siamesa + Triplet Loss"))):
    Z = PCA(n_components=2, random_state=SEED).fit_transform(E)
    for c, name in enumerate(own_names):
        ax.scatter(*Z[y_own == c].T, label=name, s=45, alpha=.8)
    ax.set_title(f"PCA de embeddings — {title}"); ax.grid(alpha=.3)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); savefig("08_espacio_embeddings"); plt.show()

### 8.6 Ejemplos de comparación de rostros

In [ ]:
rng = np.random.default_rng(7)
pairs = []
for _ in range(4):                                     # 4 pares positivos y 4 negativos
    c = rng.choice(np.unique(y_own)); i, j = rng.choice(np.where(y_own == c)[0], 2, replace=False); pairs.append((i, j))
    c1, c2 = rng.choice(np.unique(y_own), 2, replace=False)
    pairs.append((rng.choice(np.where(y_own == c1)[0]), rng.choice(np.where(y_own == c2)[0])))

fig, axes = plt.subplots(2, 8, figsize=(16, 4.8))
for col, (i, j) in enumerate(pairs):
    d = float(((E_own[i] - E_own[j]) ** 2).sum()); same = d < TAU; truth = y_own[i] == y_own[j]
    axes[0, col].imshow(X_own[i].astype("uint8")); axes[1, col].imshow(X_own[j].astype("uint8"))
    color = "green" if same == truth else "red"
    axes[0, col].set_title(f"d={d:.2f}\n{'MISMA' if same else 'DISTINTA'}", color=color, fontsize=10)
    axes[1, col].set_xlabel(f"real: {'misma' if truth else 'distinta'}", fontsize=9)
    for r in (0, 1): axes[r, col].set_xticks([]); axes[r, col].set_yticks([])
plt.suptitle(f"Verificación de pares (umbral τ={TAU:.2f}; verde = acierto, rojo = error)")
plt.tight_layout(); savefig("09_ejemplos_pares"); plt.show()

## 9. Función de verificación para fotos nuevas
Aplica el mismo pipeline de preprocesamiento a dos fotos cualesquiera y decide si son la misma persona.
Sirve para probar con una foto tomada en el momento (por ejemplo, subiéndola con `files.upload()`).

In [ ]:
def verify_faces(path_a, path_b, model=embedding_net, tau=TAU, show=True):
    face_a, info_a = preprocess_face(path_a)
    face_b, info_b = preprocess_face(path_b)
    if face_a is None or face_b is None:
        raise ValueError(f"No se detectó rostro: {path_a if face_a is None else path_b}")
    ea, eb = embed(model, np.stack([face_a, face_b]).astype("float32"))
    d = float(((ea - eb) ** 2).sum())
    same = d < tau
    if show:
        fig, axes = plt.subplots(1, 2, figsize=(5, 2.8))
        for ax, f in zip(axes, (face_a, face_b)): ax.imshow(f); ax.axis("off")
        plt.suptitle(f"d = {d:.3f}  →  {'MISMA PERSONA' if same else 'PERSONAS DISTINTAS'}",
                     color="green" if same else "firebrick"); plt.show()
    return {"distancia": d, "umbral": tau, "misma_persona": bool(same)}

def identify_face(path, model=embedding_net, tau=TAU):
    """Identificación 1:N contra la galería (prototipo de todas las fotos de cada persona)."""
    face, _ = preprocess_face(path)
    if face is None:
        raise ValueError("No se detectó rostro")
    e = embed(model, face[None].astype("float32"))[0]
    protos = np.stack([l2n(E_own[y_own == c].mean(0)) for c in range(len(own_names))])
    d = ((protos - e) ** 2).sum(-1)
    best = int(d.argmin())
    return {"identidad": own_names[best] if d[best] < tau else "DESCONOCIDO",
            "distancias": dict(zip(own_names, np.round(d, 3)))}

# Ejemplo con dos fotos originales del dataset (cambiar por fotos nuevas)
ex = ok.groupby("persona").head(2)
print(verify_faces(os.path.join(RAW_DIR, ex.iloc[0].persona, ex.iloc[0].archivo),
                   os.path.join(RAW_DIR, ex.iloc[1].persona, ex.iloc[1].archivo)))
print(verify_faces(os.path.join(RAW_DIR, ex.iloc[0].persona, ex.iloc[0].archivo),
                   os.path.join(RAW_DIR, ex.iloc[2].persona, ex.iloc[2].archivo)))

In [ ]:
# (Opcional) Probar con una foto nueva tomada en el momento
# from google.colab import files
# nueva = next(iter(files.upload()))
# print(identify_face(nueva))

## 10. Resumen de resultados y exportación
Se guardan métricas, figuras y el modelo en `/content/outputs` y se descarga un `.zip` para adjuntar al informe
(las figuras **no** contienen datos sensibles más allá de los rostros con consentimiento; revisar antes de publicar).

In [ ]:
summary = {
    "fecha": time.strftime("%Y-%m-%d %H:%M"),
    "tensorflow": tf.__version__, "keras": keras.__version__,
    "hiperparametros": {"IMG_SIZE": IMG_SIZE, "EMB_DIM": EMB_DIM, "MARGIN": MARGIN, "BATCH_SIZE": BATCH_SIZE,
                        "EPOCHS_ejecutadas": len(h), "LR": LR, "FINE_TUNE_FROM": FINE_TUNE_FROM},
    "dataset_propio": {"personas": len(own_names), "imagenes_validas": int(len(X_own)),
                       "descartadas": int((~prep.ok).sum())},
    "lfw": {"train_imgs": int(len(X_tr)), "val_imgs": int(len(X_va))},
    "entrenamiento_final": h.iloc[-1].round(4).to_dict(),
    "umbral_tau": round(float(TAU), 4),
    "verificacion_dataset_propio": results_ver.round(4).to_dict(orient="index"),
    "few_shot": fs.round(4).to_dict(orient="records"),
    "conjunto_abierto": open_df.round(4).to_dict(orient="records"),
}
with open(f"{OUT_DIR}/resultados.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False, default=float)
print(json.dumps(summary, indent=2, ensure_ascii=False, default=float))

In [ ]:
import shutil
shutil.make_archive("/content/outputs_redes_neuronales_v", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("/content/outputs_redes_neuronales_v.zip")
except Exception as e:
    print("Descarga manual desde el panel de archivos:", e)

## 11. Análisis y conclusiones (completar con los resultados obtenidos)

* **Efecto del entrenamiento métrico:** comparar AUC y exactitud One-shot de la línea base ImageNet frente a la red
  siamesa. Las características ImageNet describen objetos, no identidades; Triplet Loss reorganiza el espacio para que
  la distancia refleje identidad.
* **One-shot vs Few-shot:** la exactitud suele aumentar con *k* porque el prototipo promedia variaciones de pose e
  iluminación.
* **Zero-shot / identidades no vistas:** el dataset propio nunca se usó para entrenar; un AUC alto indica que el
  espacio de embeddings generaliza a personas nuevas.
* **Errores:** revisar los pares en rojo y las fotos descartadas o borrosas; suelen deberse a perfiles extremos,
  oclusiones (gafas de sol, tapabocas) o iluminación muy desigual.
* **Limitaciones:** dataset propio pequeño, posible sesgo demográfico de LFW, tripletas aleatorias (la minería de
  tripletas semi-difíciles mejoraría la convergencia).
* **Consideraciones éticas:** consentimiento informado, no publicar fotos en el repositorio, uso exclusivamente académico.